# HDS-M03 Mini Project: Lung Segmentation from Chest X-rays

## Part 4: Classification (step 9)

TB changes the *texture* of the lung (patchy, nodular areas). The **Gray-Level Co-occurrence Matrix
(GLCM)** counts how often a pixel of grey level $i$ sits next to a pixel of grey level $j$, and from
this table we get summary features (see the `biomedia_features_05_GLCM` practical):

| Feature | A high value means |
|---|---|
| contrast | large brightness jumps between neighbours |
| dissimilarity | like contrast, but grows linearly |
| homogeneity | neighbours have similar grey levels |
| energy | very repetitive / uniform texture |
| correlation | neighbouring grey levels are linearly related |

The features are computed **only inside the lungs** (see `lungseg.glcm_features` for how).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# local helper module, all repeated code lives in lungseg.py
import lungseg

# create a folder for figures if it doesn't exist
os.makedirs('figures', exist_ok=True)

import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# create a folder for result tables if it doesn't exist
os.makedirs('results', exist_ok=True)

# set a constant seed for reproducibility
SEED = 42

#### (9a) Calculating GLCM features

With `USE_PREDICTED_MASK = False` the radiologist's lung mask is used, so the classifier is judged on
texture alone. Setting it to `True` uses my own segmentation instead, giving a fully automatic pipeline.

In [ ]:
# False = ground-truth lungs, True = our own segmentation
USE_PREDICTED_MASK = False

# e.g. 10 for a quick test, None for all images
N_PER_CLASS = None

cases = lungseg.list_cases(0)[:N_PER_CLASS] + lungseg.list_cases(1)[:N_PER_CLASS]

rows = []

for name in cases:

    image, gt = lungseg.load_case(name)
    mask      = lungseg.segment_lungs(image) if USE_PREDICTED_MASK else gt

    # dictionary of the 5 GLCM features, plus file name and label (0 = normal, 1 = TB)
    feats          = lungseg.glcm_features(image, mask)
    feats['file']  = name
    feats['label'] = lungseg.get_label(name)

    rows.append(feats)

features = pd.DataFrame(rows)
features.to_csv('results/glcm_features.csv', index=False)
features.head()

#### (9b) Comparing the two classes

First a scatter plot of two features, coloured by class:

In [ ]:
# try other pairs from lungseg.GLCM_PROPS
x_feat, y_feat = 'contrast', 'homogeneity'

plt.figure(figsize=(6, 5))

for label, colour, text in [(0, 'tab:blue', 'normal'), (1, 'tab:red', 'TB')]:
    subset = features[features['label'] == label]
    plt.scatter(subset[x_feat], subset[y_feat], c=colour, label=text, alpha=0.7)

plt.xlabel(f'GLCM {x_feat}')
plt.ylabel(f'GLCM {y_feat}')
plt.title('GLCM features inside the lungs')
plt.legend()
plt.savefig('figures/fig_4_glcm_scatter.pdf', bbox_inches='tight')
plt.show()

Then, for every feature, the median in each class and a Mann-Whitney U test p-value. A small p
(e.g. < 0.05) is evidence that the feature differs between normal and TB.

In [ ]:
stats = []

for feat in lungseg.GLCM_PROPS:

    normal_vals = features.loc[features['label'] == 0, feat]
    tb_vals     = features.loc[features['label'] == 1, feat]

    stats.append({
        'feature'       : feat,
        'median normal' : normal_vals.median(),
        'median TB'     : tb_vals.median(),
        'p-value'       : mannwhitneyu(normal_vals, tb_vals).pvalue,
    })

pd.DataFrame(stats).round(4)

Note that testing 5 features means 5 separate tests, so with a 0.05 cut-off some false positives are
expected by chance. A Bonferroni correction (multiplying each p-value by 5) is the simplest fix.

#### (9c) Developing a classifier

I use logistic regression:

- `StandardScaler` puts every feature on the same scale (mean 0, standard deviation 1), which logistic
  regression needs when features have very different ranges.
- With only 138 images, I use **stratified 5-fold cross-validation**: the model trains on 4/5 of the
  data, predicts the remaining 1/5, and this rotates 5 times so every image is predicted by a model
  that never saw it. *Stratified* means each fold keeps the same normal/TB ratio.

In [ ]:
# inputs: one row per image, one column per feature; targets: 0 / 1
X = features[lungseg.GLCM_PROPS].values
y = features['label'].values

model = make_pipeline(StandardScaler(), LogisticRegression())
cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# out-of-fold predicted labels and probabilities of TB
y_pred = cross_val_predict(model, X, y, cv=cv)
y_prob = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]

print(classification_report(y, y_pred, target_names=['normal', 'TB']))
print('Confusion matrix (rows = truth, columns = prediction):')
print(confusion_matrix(y, y_pred))
print(f'\nROC AUC: {roc_auc_score(y, y_prob):.3f}  (0.5 = guessing, 1.0 = perfect)')

Points for the write-up:

- Which features differ most between normal and TB, and does that match what TB looks like?
- How does the classifier compare to always guessing "normal" (80/138 ≈ 58% accuracy)?
- Sensitivity (recall for TB) matters most for screening: how many TB cases were missed?
- Limitations: small data set, whole-lung averages dilute small lesions, GLCM settings (`levels`, `distances`) were not tuned.
- Extensions: set `USE_PREDICTED_MASK = True`, add simple intensity features (mean, standard deviation),
  or try an SVM (`from sklearn.svm import SVC`) as in the `biomedia_classification_05` practical.